# Практична робота №9 — Динамічне програмування: принцип оптимальності БеллманаВкажи номер варіанта, коли клітинка попросить — решта коду підхопить твою таблицю прибутків підприємств, обсяг ресурсу R та кількість етапів власної мережі автоматично.

In [ ]:
import numpy as np
import pandas as pd
import itertools
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
from variants import VARIANTS

_default = globals().get("VARIANT", 1)
_raw = input(f"Номер варіанта (1-{len(VARIANTS)}) [{_default}]: ").strip()
VARIANT = int(_raw) if _raw else _default
assert VARIANT in VARIANTS, f"Немає варіанта {VARIANT}"

## 1. Формалізація задачі розподілу ресурсуЗадача: розподілити R одиниць ресурсу між N = 4 підприємствами так, щоб сумарний прибуток був максимальним.- **Крок** k = 1, 2, 3, 4 — черговість, у якій "обслуговується" k-те підприємство (порядок умовний, на кінцевий результат не впливає).- **Стан** sₖ — залишок нерозподіленого ресурсу перед кроком k (sₖ = 0 .. R).- **Керування** uₖ — скільки одиниць ресурсу виділити k-му підприємству, 0 ≤ uₖ ≤ sₖ.- **Функція переходу**: sₖ₊₁ = sₖ − uₖ.- **Виграш кроку**: gₖ(uₖ) — табличний прибуток k-го підприємства від вкладення uₖ одиниць.

In [ ]:
g, R, STAGES = VARIANTS[VARIANT]
g = np.array(g, float)
N = g.shape[0]
print(f"Варіант {VARIANT}")
print("Прибутки підприємств (рядки) від вкладень u = 0..5 (стовпці):")
print(g)
print(f"R = {R}, N = {N} підприємства, кількість етапів власної мережі (частина 2) = {STAGES}")

## 2. Рекурентне співвідношення БеллманаФункція Беллмана Fₖ(s) — максимальний сумарний прибуток перших k підприємств, якщо в розпорядженні є s одиниць ресурсу:**Fₖ(s) = max₍₀ ≤ u ≤ s₎ [ gₖ(u) + Fₖ₋₁(s − u) ],  F₀(s) = 0.**Разом зі значенням Fₖ(s) на кожному кроці запам'ятовують **умовне оптимальне керування** uₖ*(s) — те u, на якому досягається максимум. Зворотний хід виконується від k=1 до k=N; після нього прямий хід (нижче) відновлює фактичний план.

In [ ]:
F = np.zeros((N + 1, R + 1))
U = np.zeros((N + 1, R + 1), int)
for k in range(1, N + 1):
    for s in range(R + 1):
        vals = [g[k - 1, u] + F[k - 1, s - u] for u in range(s + 1)]
        U[k, s] = int(np.argmax(vals))
        F[k, s] = max(vals)

df_F = pd.DataFrame(F[1:], index=[f"F_{k}" for k in range(1, N + 1)], columns=[f"s={s}" for s in range(R + 1)])
df_F

In [ ]:
df_U = pd.DataFrame(U[1:], index=[f"u_{k}*" for k in range(1, N + 1)], columns=[f"s={s}" for s in range(R + 1)])
df_U

## 3. Пояснення обчислення однієї клітинки вручнуПокажемо детально, як рахується Fₖ(s) для k = 2, s = R: перебираємо всі можливі uₖ від 0 до s, для кожного беремо вже обчислене на попередньому кроці F₁(s − u) і додаємо gₖ(u), після чого обираємо максимум.

In [ ]:
k_demo, s_demo = 2, R
vals_demo = [(u, g[k_demo - 1, u], F[k_demo - 1, s_demo - u]) for u in range(s_demo + 1)]
for u, gu, fprev in vals_demo:
    print(f"u={u}: g_{k_demo}({u})={gu:.0f} + F_{k_demo-1}({s_demo-u})={fprev:.0f}  ->  {gu+fprev:.0f}")
print(f"F_{k_demo}({s_demo}) = max(...) = {F[k_demo, s_demo]:.0f}, досягається при u*_{k_demo}({s_demo}) = {U[k_demo, s_demo]}")

## 4. Прямий хід — відновлення оптимального плануПрямий хід рухається від останнього кроку до першого: беремо повний ресурс s = R, зчитуємо u_N*(s), зменшуємо залишок на цю величину, переходимо до кроку N−1, і так далі.

In [ ]:
plan = []
s = R
for k in range(N, 0, -1):
    u_k = int(U[k, s])
    plan.append(u_k)
    s -= u_k
plan.reverse()
total_profit = float(F[N, R])
print(f"Оптимальний план розподілу (u1, u2, u3, u4) = {tuple(plan)}")
print(f"Сумарний прибуток: {total_profit:.0f}")

## 5. Порівняння з рівномірним і жадібним розподілами

In [ ]:
base = R // N
rem = R - base * N
uniform_profit = float(sum(g[i, base] for i in range(N)))

greedy_vals = [float(g[i, R]) for i in range(N)]
greedy_best = int(np.argmax(greedy_vals))
greedy_profit = greedy_vals[greedy_best]

gain_uniform = (total_profit - uniform_profit) / uniform_profit * 100
gain_greedy = (total_profit - greedy_profit) / greedy_profit * 100

print(f"ДП:          план {tuple(plan)}, прибуток {total_profit:.0f}")
print(f"Рівномірний: по {base} кожному (залишок {rem} не використано), прибуток {uniform_profit:.0f}")
print(f"Жадібний:    усе підприємству {greedy_best + 1}, прибуток {greedy_profit:.0f}")
print(f"Виграш ДП проти рівномірного: {gain_uniform:.2f}%")
print(f"Виграш ДП проти жадібного:    {gain_greedy:.2f}%")

## 6. Перевірка повним переборомПростір станів тут невеликий (композиції R на N частин), тому оптимум ДП можна звірити з результатом повного перебору всіх допустимих планів.

In [ ]:
best_bf = -1.0
best_plan_bf = None
for combo in itertools.product(range(R + 1), repeat=N):
    if sum(combo) != R:
        continue
    val = float(sum(g[i, combo[i]] for i in range(N)))
    if val > best_bf:
        best_bf, best_plan_bf = val, combo
print(f"Повний перебір: оптимальний план {best_plan_bf}, прибуток {best_bf:.0f}")
assert abs(best_bf - total_profit) < 1e-9, "ДП і повний перебір мають збігатись!"
print("Збігається з результатом ДП.")

## 7. Власна багатоетапна мережа і пошук найкоротшого шляхуБудуємо орієнтовану мережу без циклів із заданою у варіанті кількістю етапів (STAGES) і не менше 9 вершин. Вага ребра — довжина ділянки шляху. Функція Беллмана V(v) — довжина найкоротшого шляху від вершини v до кінцевої вершини Z:**V(Z) = 0,  V(v) = min₍(v,w)∈E₎ [ c(v, w) + V(w) ].**Обчислення — у зворотному топологічному порядку (від Z до A).

In [ ]:
def layer_sizes(stages):
    """Шари мережі так, щоб вершин було не менше 9 (як у прикладі з методички)."""
    presets = {3: [1, 3, 4, 1], 4: [1, 2, 3, 2, 1], 5: [1, 2, 2, 2, 1, 1]}
    return presets[stages]


def build_network(stages, variant):
    sizes = layer_sizes(stages)
    letters = "BCDEFGHIJKLMNOPQRSTUVWXY"
    layers = []
    for li, sz in enumerate(sizes):
        if li == 0:
            layers.append(["A"])
        elif li == len(sizes) - 1:
            layers.append(["Z"])
        else:
            layers.append([f"{letters[li - 1]}{j + 1}" for j in range(sz)])
    edges = {v: {} for layer in layers for v in layer}
    for li in range(len(layers) - 1):
        for i, u in enumerate(layers[li]):
            for j, v in enumerate(layers[li + 1]):
                w = ((i + 1) * 3 + (j + 1) * 5 + variant * 2 + li * 7) % 12 + 1
                edges[u][v] = w
    return layers, edges


layers, edges = build_network(STAGES, VARIANT)
n_vertices = sum(len(l) for l in layers)
print(f"Етапів: {STAGES}, вершин у мережі: {n_vertices}")
for layer in layers:
    print(layer)

In [ ]:
all_nodes = [v for layer in layers for v in layer]
start, end = layers[0][0], layers[-1][0]
rev_order = list(reversed(all_nodes))

V = {end: 0.0}
nxt = {}
for v in rev_order[1:]:
    V[v], nxt[v] = min(((edges[v][w] + V[w], w) for w in edges[v]), key=lambda t: t[0])

path = [start]
v = start
while v != end:
    v = nxt[v]
    path.append(v)

print("Функції Беллмана V(v):")
for v in reversed(all_nodes):
    print(f"  V({v}) = {V[v]:.0f}")
print(f"Найкоротший шлях: {' -> '.join(path)}, довжина = {V[start]:.0f}")

In [ ]:
def all_paths(u, target):
    if u == target:
        yield [u]
        return
    for w in edges[u]:
        for tail in all_paths(w, target):
            yield [u] + tail


best_len_bf = min(sum(edges[p[i]][p[i + 1]] for i in range(len(p) - 1)) for p in all_paths(start, end))
print(f"Повний перебір усіх шляхів A -> Z: мінімальна довжина = {best_len_bf}")
assert best_len_bf == V[start]
print("Збігається з результатом ДП.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
pos = {}
for li, layer in enumerate(layers):
    for j, vv in enumerate(layer):
        y = j - (len(layer) - 1) / 2
        pos[vv] = (li, y)
for u in edges:
    for vv, w in edges[u].items():
        x1, y1 = pos[u]; x2, y2 = pos[vv]
        on_path = u in path and vv in path and path.index(vv) == path.index(u) + 1
        ax.plot([x1, x2], [y1, y2], color='red' if on_path else 'gray',
                lw=2.5 if on_path else 1, zorder=1 if on_path else 0)
        ax.annotate(str(w), ((x1 + x2) / 2, (y1 + y2) / 2), fontsize=8, color='black')
for vv, (x, y) in pos.items():
    ax.scatter([x], [y], s=500, color='lightgreen' if vv in path else 'lightblue', zorder=2, edgecolors='black')
    ax.annotate(vv, (x, y), ha='center', va='center', fontsize=9, zorder=3)
ax.set_title(f"Варіант {VARIANT}: мережа ({STAGES} етапи) і найкоротший шлях {' -> '.join(path)}")
ax.axis('off')
fig.tight_layout()
fig.savefig('network.png', dpi=150)
fig

## 8. Додаткова задача: заміна обладнання за принципом БеллманаМодель (спільна для всіх варіантів, як окрема ілюстрація методу): машина має вік a = 0..4 роки, щороку приносить прибуток p(a), вимагає обслуговування c(a); на початку кожного з 5 років горизонту приймається рішення — **"зберегти"** (вік зростає на 1, дохід року = p(a) − c(a)) або **"замінити"** (сплатити вартість заміни, машина одразу стає новою, дохід року = p(0) − c(0) − вартість заміни, вік після заміни = 1). У віці 4 заміна обов'язкова. Fₖ(a) — максимальний прибуток за k років, що залишились, якщо на початку цього періоду вік машини a.

In [ ]:
p_age = [100, 90, 75, 55, 30]   # прибуток за рік залежно від віку 0..4
c_age = [10, 15, 25, 40, 60]    # витрати на обслуговування залежно від віку
REPLACE_COST = 50
HORIZON = 5
MAX_AGE = 4


def revenue_keep(a):
    return p_age[a] - c_age[a]


revenue_replace = p_age[0] - c_age[0] - REPLACE_COST

Veq = {0: {a: 0.0 for a in range(MAX_AGE + 1)}}
dec = {}
for k in range(1, HORIZON + 1):
    Veq[k] = {}
    dec[k] = {}
    for a in range(MAX_AGE + 1):
        options = []
        if a < MAX_AGE:
            options.append(("зберегти", revenue_keep(a) + Veq[k - 1][a + 1]))
        options.append(("замінити", revenue_replace + Veq[k - 1][1]))
        dec[k][a], Veq[k][a] = max(options, key=lambda t: t[1])

a = 0
plan_eq = []
total_eq = 0.0
for k in range(HORIZON, 0, -1):
    d = dec[k][a]
    gain = revenue_keep(a) if d == "зберегти" else revenue_replace
    plan_eq.append((HORIZON - k + 1, a, d, gain))
    total_eq += gain
    a = a + 1 if d == "зберегти" else 1

print("рік | вік на початку | рішення  | прибуток року")
for year, age, d, gain in plan_eq:
    print(f"{year:>3} | {age:>14} | {d:<8} | {gain:>5.0f}")
print(f"Сумарний прибуток за {HORIZON} років: {total_eq:.0f}")

In [ ]:
best_eq = -1.0
best_seq = None
for seq in itertools.product([0, 1], repeat=HORIZON):
    age = 0
    total_b = 0.0
    ok = True
    for d in seq:
        if age == MAX_AGE and d == 0:
            ok = False
            break
        if d == 0:
            total_b += revenue_keep(age)
            age += 1
        else:
            total_b += revenue_replace
            age = 1
    if ok and total_b > best_eq:
        best_eq, best_seq = total_b, seq
print(f"Повний перебір (2^5 послідовностей): {best_eq:.0f}, keep(0)/replace(1) = {best_seq}")
assert abs(best_eq - total_eq) < 1e-9
print("Збігається з результатом ДП.")

## 9. Обчислювальна складність ДП проти повного переборуДля задачі розподілу R одиниць між N підрозділами: ДП потребує O(N·R²) операцій, повний перебір — O(R^N) (точніше, C(R+N−1, N−1) варіантів розподілу).

In [ ]:
Rs = list(range(5, 31, 5))
dp_ops = [N * (Rv + 1) * (Rv + 2) // 2 for Rv in Rs]
brute_ops = [math.comb(Rv + N - 1, N - 1) for Rv in Rs]

df_complexity = pd.DataFrame({"R": Rs, "ДП (операцій)": dp_ops, "Повний перебір (варіантів)": brute_ops})
df_complexity

In [ ]:
fig2, ax2 = plt.subplots(figsize=(7, 5))
ax2.plot(Rs, dp_ops, 'o-', label='ДП: N*(R+1)*(R+2)/2')
ax2.plot(Rs, brute_ops, 's-', label='Повний перебір: C(R+N-1, N-1)')
ax2.set_yscale('log')
ax2.set_xlabel('R (обсяг ресурсу)')
ax2.set_ylabel('Кількість обчислень (лог. шкала)')
ax2.set_title(f'Варіант {VARIANT}: зростання трудомісткості при N={N}')
ax2.legend()
fig2.tight_layout()
fig2.savefig('complexity.png', dpi=150)
fig2

## 10. Додаткове завдання (на «відмінно»): R = 10 з лінійно продовженими таблицями прибуткуПродовжуємо кожен рядок таблиці прибутків на u = 6..10, використовуючи останній граничний прирост (різницю між двома останніми відомими значеннями) як сталий крок, і розв'язуємо ту саму задачу для R = 10.

In [ ]:
def extend_linear(row, new_len):
    row = list(row)
    last_diff = row[-1] - row[-2]
    while len(row) < new_len:
        row.append(row[-1] + last_diff)
    return row


R_bonus = 10
g_bonus = np.array([extend_linear(g[i], R_bonus + 1) for i in range(N)], float)

F_b = np.zeros((N + 1, R_bonus + 1))
U_b = np.zeros((N + 1, R_bonus + 1), int)
for k in range(1, N + 1):
    for s in range(R_bonus + 1):
        vals = [g_bonus[k - 1, u] + F_b[k - 1, s - u] for u in range(s + 1)]
        U_b[k, s] = int(np.argmax(vals))
        F_b[k, s] = max(vals)

plan_b, s_b = [], R_bonus
for k in range(N, 0, -1):
    u_k = int(U_b[k, s_b])
    plan_b.append(u_k)
    s_b -= u_k
plan_b.reverse()
print(f"R={R_bonus}: оптимальний план {tuple(plan_b)}, прибуток {F_b[N, R_bonus]:.0f}")

best_bf_b = -1.0
best_plan_bf_b = None
for combo in itertools.product(range(R_bonus + 1), repeat=N):
    if sum(combo) != R_bonus:
        continue
    val = float(sum(g_bonus[i, combo[i]] for i in range(N)))
    if val > best_bf_b:
        best_bf_b, best_plan_bf_b = val, combo
print(f"Повний перебір для R={R_bonus}: {best_plan_bf_b}, прибуток {best_bf_b:.0f}")
assert abs(best_bf_b - F_b[N, R_bonus]) < 1e-9
print("Збігається.")

## ВисновкиМетод динамічного програмування звів задачу комбінаторного вибору (O(R^N) варіантів) до послідовності з N простих одновимірних підзадач (O(N·R²) операцій), давши точний (підтверджений повним перебором) оптимум як для розподілу ресурсу, так і для пошуку найкоротшого шляху та заміни обладнання. Жадібна і рівномірна стратегії — лише наближення, що в загальному випадку поступаються ДП.